# 고장 종류별 경보 시스템 — EDA 결정 반영

| 항목 | 내용 |
|---|---|
| 질문 | EDA에서 정한 방향(셔터 1, 평소 대비 z값, 고장 종류별 접근)대로 탐지기를 나누고 **직후 경보 규칙과 합치면**, 현장 조건에서 고장을 몇 건이나 미리 잡는가? |
| 종류 | 압력 저하·누설: **지도학습 (분류, 로지스틱 회귀)** / 압력 과다: **비지도 (IsolationForest, 정답은 채점에만 사용)** / 직후 경보: **규칙 (학습 없음)** |
| 정답(라벨) | 지도 탐지기: 고장 종류별로 "라벨 창 안에 그 고장이 나는가" (예=1). 압력 저하·누설 24시간 |
| 데이터 | `data/processed/step/` (레시피 스텝 구간 = 한 행), 셔터 1(식각 중)만 |
| 분할 | `field_eval.ipynb`와 같은 **같은 장비의 시간 분할**. 장비마다 앞 75% = 학습·보정, 뒤 25% = 평가 |
| 이전 결과 | `field_eval.ipynb`: 가장 나은 단일 모델의 탐지율 35~47%, 정밀도 10% 미만 |

## 1. EDA에서 가져온 결정

| EDA 결과 (`eda/eda_step.ipynb`) | 이 노트북에서 |
|---|---|
| 센서 분산의 93%가 장비·레시피·스텝·셔터 차이 (4-2) | 장비·레시피·스텝별 **평소 대비 z값** (셔터는 1로 고정). 기준은 학습 구간 평소 행에서만 구한다 |
| 식각 중과 아닐 때는 다른 상태, 셔터 1에서 구분력이 더 큼 (4-4, 4-5) | **셔터 1 구간만** 쓴다 |
| 뚜렷한 신호는 냉각 압력, 압력 과다는 냉각 유량 (4-5, 4-7c) | 입력 센서는 **냉각 압력·냉각 유량 두 개만**. 회전·누적 카운터는 쓰지 않는다 |
| 압력 저하·누설: 일주일 전부터 이어지는 **수준 차이** (4-7c) | 지난 24시간·7일 평균과 **최근 대비 변화**(24시간 평균 − 7일 평균) |
| 압력 과다: 냉각 유량이 **약 3~6시간 전부터 서서히** 떨어짐 (4-7c) | 지난 6시간 평균과 6시간 변화(6시간 평균 − 7일 평균), 정답 없이 "평소와 다름"을 잡는 이상 감지 |
| 같은 고장이 하루 안에 반복 기록됨, 직전 기록 없는 고장 30~60% (4-8, 4-9) | **하루 묶음 사건**으로 채점하고, **직후 경보 규칙**으로 반복 고장을 잡는다. "판단 가능 사건 중 탐지율"을 함께 적는다 |
| 장비마다 수준·방향이 다름 (부록 `eda_tools.ipynb`) | 경보 기준을 **장비마다 따로** 정한다 |

## 2. 구성

```
스텝 구간(셔터 1) → ① 장비별 평소 기준(z값) → ② 고장 종류별 탐지기 ┐
                                              ③ 직후 경보 규칙 ─────┤→ ④ 경보 합치기 → ⑤ 채점
```

| 고장 | ② 탐지기 | 입력 |
|---|---|---|
| 압력 저하 | 로지스틱 회귀 (지도, 24시간 라벨) | 냉각 압력 z(평균·최소·최대), 6시간·24시간·7일 평균, 최근 변화 2개 / 냉각 유량 z 평균·24시간 평균·변화 / 고장 이력 5개 |
| 압력 과다 | IsolationForest (비지도) | 냉각 압력·유량 z 평균·최소, 6시간 평균, 6시간 변화 |
| 누설 | 로지스틱 회귀 (지도, 24시간 라벨) | 냉각 압력 24시간·7일 평균, 최근 변화 / 고장 이력 3개 |

- **③ 직후 경보 규칙**: 어떤 종류든 고장이 기록되면 그 시각에 경보.
- **④ 경보 합치기**: 탐지기 경보와 규칙 경보를 모은 뒤, 경보 뒤 24시간은 다시 울리지 않는다.
- **경보 기준**: 장비마다 보정 구간에서 탐지기 오경보가 **주 1회 이하**가 되는 가장 낮은 점수. 5-2에서 허용량을 바꿔 가며 본다.
- **채점**: 사건(같은 종류 기록을 1일 간격으로 묶음)의 첫 기록 전 24시간 안에 경보가 있으면 탐지. 경보 뒤 24시간 안에 그 종류 고장이 없으면 오경보.
- 고장 이름의 ` [NoWaferID]` 표기는 지우고 같은 종류로 센다(EDA와 같음). `field_eval`은 이 표기를 다른 이름으로 봐서 사건 수가 조금 다를 수 있다.

In [1]:
from pathlib import Path
import json
import time
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from IPython.display import display
warnings.filterwarnings('ignore')

cwd = Path.cwd().resolve()
ROOT = next(path for path in (cwd, cwd.parent) if (path / 'data/processed/step').exists())
PROC = ROOT / 'data/processed'

def show(fig, height=480):
    # 그림은 파일로 저장하지 않고 노트북 출력으로만 남긴다
    fig.update_layout(template='plotly_white', height=height, font=dict(family='Malgun Gothic, sans-serif'),
                      margin=dict(l=30, r=30, t=75, b=60))
    display({'application/vnd.plotly.v1+json': json.loads(fig.to_json())}, raw=True)

# ── 상수 ─────────────────────────────────────────────────────────
SENSORS = ['FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE']      # EDA: 냉각 압력·유량만 신호가 있음
STATS = ['mean', 'min', 'max']
FAULTS = ['FlowCool Pressure Dropped Below Limit', 'Flowcool Pressure Too High Check Flowcool Pump', 'Flowcool leak']
FSHORT = dict(zip(FAULTS, ['압력 저하', '압력 과다', '누설']))
TTF = {fault: 'TTF_' + fault for fault in FAULTS}
HOUR, DAY, WEEK = 3600, 86400, 7 * 86400
ALARM_WINDOW = DAY            # 경보 뒤 이 시간 안에 고장 = 맞은 경보, 사건 전 이 시간 안 경보 = 탐지
FA_TARGET = 1.0               # 기본 오경보 허용량 (장비 1대, 주당)
MIN_NORMAL_SEGMENTS = 10      # 평소 기준을 만들 최소 구간 수 (학습 구간만 쓰므로 EDA의 30보다 낮춤)
SEED = 0

# ── 불러오기 ──────────────────────────────────────────────────────
load_cols = ['Tool', 'time_start', 'time_end', 'recipe', 'recipe_step', 'shutter', *TTF.values(),
             *[f'{sensor}_{stat}' for sensor in SENSORS for stat in STATS]]
segments = pd.concat([pd.read_parquet(path, columns=load_cols) for path in sorted((PROC / 'step').glob('*.parquet'))],
                     ignore_index=True)
segments['Tool'] = segments.Tool.astype(str)

faults = pd.read_csv(PROC / 'faults.csv')
faults['Tool'] = faults.Tool.map(lambda raw_name: f'{raw_name[2:]}_{raw_name[:2]}')        # 01M01 → M01_01
faults['fault_type'] = faults.fault_name.str.replace(' [NoWaferID]', '', regex=False)
faults = faults.drop_duplicates(['Tool', 'time', 'fault_type']).sort_values(['Tool', 'time']).reset_index(drop=True)
assert set(faults.Tool) <= set(segments.Tool) and set(faults.fault_type) == set(FAULTS)

# ── 장비별 기간 나누기: 학습(fit) 2/3 · 보정(calib) 1/3 · 간격 1일 · 평가(test) 뒤 25% ──
tool_span = segments.groupby('Tool').agg(t_first=('time_start', 'min'), t_last=('time_end', 'max'))
tool_span['test_start'] = tool_span.t_first + 0.75 * (tool_span.t_last - tool_span.t_first)
tool_span['fit_end'] = tool_span.t_first + (tool_span.test_start - DAY - tool_span.t_first) * 2 / 3

# 평소 여부: 가장 가까운 고장 기록(종류 무관)까지 24시간 초과
nearest_fault_seconds = np.full(len(segments), np.inf)
for tool, row_idx in segments.groupby('Tool').indices.items():
    fault_times = faults.loc[faults.Tool == tool, 'time'].to_numpy()
    seg_start, seg_end = segments.time_start.to_numpy()[row_idx], segments.time_end.to_numpy()[row_idx]
    prev_pos, next_pos = np.searchsorted(fault_times, seg_start), np.searchsorted(fault_times, seg_end)
    since_prev = np.where(prev_pos > 0, seg_start - fault_times[np.maximum(prev_pos - 1, 0)], np.inf)
    until_next = np.where(next_pos < len(fault_times), fault_times[np.minimum(next_pos, len(fault_times) - 1)] - seg_end, np.inf)
    nearest_fault_seconds[row_idx] = np.minimum(since_prev, until_next)
segments['is_normal'] = nearest_fault_seconds > DAY

# 셔터 1(식각 중)만 남기고 기간 표시
segments = segments[segments.shutter == 1].join(tool_span, on='Tool').sort_values(['Tool', 'time_start']).reset_index(drop=True)
segments['part'] = np.select([segments.time_end < segments.fit_end, segments.time_end < segments.test_start - DAY,
                              segments.time_start >= segments.test_start], ['fit', 'calib', 'test'], 'gap')
part = segments.part.to_numpy()
is_fit, is_calib, is_test = part == 'fit', part == 'calib', part == 'test'
tool_of_row = segments.Tool.to_numpy()
start_of_row = segments.time_start.to_numpy()

part_table = segments.groupby('part').agg(구간=('Tool', 'size'), 장비=('Tool', 'nunique')).reindex(['fit', 'calib', 'gap', 'test'])
part_table['비율(%)'] = (part_table.구간 / part_table.구간.sum() * 100).round(1)
display(part_table)

,구간,장비,비율(%)
part,,,
fit,278002,20,52.8
calib,120259,20,22.9
gap,1236,20,0.2
test,126769,20,24.1


## 3. 입력 만들기 (그 시점까지 알 수 있는 값만)

- **① 평소 기준**: 장비·레시피·스텝별로 **학습 구간의 평소 행**에서 평균·표준편차를 구한다. 평가 구간 정보는 기준에 들어가지 않는다. 최소·최대 z도 평균의 기준으로 나눈다. ±10에서 자른다(EDA 3-3 처리 방안).
- **이동 평균**: 장비마다 시각 순으로 지난 6시간·24시간·7일의 z 평균(행 3개 이상일 때). 그 행까지의 과거만 쓴다.
- **최근 변화**: 24시간 평균 − 7일 평균(수준 변화), 6시간 평균 − 7일 평균(빠른 변화).
- **고장 이력**: 그 행 시작 전에 기록된 고장만 쓴다.

In [2]:
MEAN_COLS = [sensor + '_mean' for sensor in SENSORS]
segments['step_key'] = segments.recipe_step.fillna(-1)
baseline_keys = ['Tool', 'recipe', 'step_key']

# ① 장비·레시피·스텝별 평소 기준 (학습 구간 평소 행만)
baseline_rows = segments[is_fit & segments.is_normal.to_numpy()]
grouped_baseline = baseline_rows.groupby(baseline_keys)[MEAN_COLS]
normal_segment_count = grouped_baseline.size()
baseline_mean, baseline_std = grouped_baseline.mean(), grouped_baseline.std()
enough_normal = np.broadcast_to((normal_segment_count >= MIN_NORMAL_SEGMENTS).to_numpy()[:, None], baseline_std.shape)
baseline_std = baseline_std.where((baseline_std > 1e-6) & enough_normal)        # 기준이 약한 조합은 z를 비운다
row_key = pd.MultiIndex.from_frame(segments[baseline_keys])
mean_by_row = baseline_mean.reindex(row_key).to_numpy()
std_by_row = baseline_std.reindex(row_key).to_numpy()

features = {}
for sensor_pos, sensor in enumerate(SENSORS):
    for stat in STATS:
        raw_values = segments[f'{sensor}_{stat}'].to_numpy()
        features[f'{sensor}_{stat}_z'] = np.clip((raw_values - mean_by_row[:, sensor_pos]) / std_by_row[:, sensor_pos], -10, 10)
features = pd.DataFrame(features).astype('float32')

# 이동 평균: 장비마다 시각 순, 지난 6시간·24시간·7일 (segments가 장비·시각 순이라 행 순서가 같다)
mean_z = features[[f'{sensor}_mean_z' for sensor in SENSORS]].copy()
mean_z.index = pd.to_datetime(segments.time_start, unit='s')
for window in ['6h', '24h', '7d']:
    rolling_mean = mean_z.groupby(tool_of_row).rolling(window, min_periods=3).mean().reset_index(level=0, drop=True)
    for column in mean_z.columns:
        features[f'{column}_r{window}'] = rolling_mean[column].to_numpy()
for column in mean_z.columns:
    features[f'{column}_chg24'] = features[f'{column}_r24h'] - features[f'{column}_r7d']   # 수준 변화
    features[f'{column}_chg6'] = features[f'{column}_r6h'] - features[f'{column}_r7d']     # 빠른 변화

# 고장 이력: 그 행 시작 시각 이전에 기록된 고장만 (시간 단위, 기록 없으면 1년으로 채움)
HISTORY_CAP_HOURS = 24 * 365.0
history = {name: np.full(len(segments), HISTORY_CAP_HOURS, dtype='float32')
           for name in ['since_any', 'since_drop', 'since_high', 'since_leak']}
history['n7d_any'] = np.zeros(len(segments), dtype='float32')
for tool, row_idx in segments.groupby('Tool').indices.items():
    row_starts = start_of_row[row_idx]
    tool_faults = faults[faults.Tool == tool]
    for name, fault_subset in [('since_any', tool_faults), ('since_drop', tool_faults[tool_faults.fault_type == FAULTS[0]]),
                               ('since_high', tool_faults[tool_faults.fault_type == FAULTS[1]]),
                               ('since_leak', tool_faults[tool_faults.fault_type == FAULTS[2]])]:
        fault_times = fault_subset.time.to_numpy()
        if len(fault_times) == 0:
            continue
        prev_pos = np.searchsorted(fault_times, row_starts, 'left')
        hours_since = (row_starts - fault_times[np.maximum(prev_pos - 1, 0)]) / HOUR
        history[name][row_idx] = np.where(prev_pos > 0, np.minimum(hours_since, HISTORY_CAP_HOURS), HISTORY_CAP_HOURS)
    all_times = tool_faults.time.to_numpy()
    history['n7d_any'][row_idx] = np.searchsorted(all_times, row_starts, 'left') - np.searchsorted(all_times, row_starts - WEEK, 'left')
for name, values in history.items():
    features[name] = values

# 고장 종류별 입력 묶음
PRESSURE, FLOW = 'FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE'
DETECTOR_INPUTS = {
    '압력 저하': [f'{PRESSURE}_{stat}_z' for stat in STATS]
               + [f'{PRESSURE}_mean_z_r{window}' for window in ['6h', '24h', '7d']]
               + [f'{PRESSURE}_mean_z_chg24', f'{PRESSURE}_mean_z_chg6',
                  f'{FLOW}_mean_z', f'{FLOW}_mean_z_r24h', f'{FLOW}_mean_z_chg24']
               + ['since_any', 'since_drop', 'since_high', 'since_leak', 'n7d_any'],
    '압력 과다': [f'{sensor}_{stat}_z' for sensor in (FLOW, PRESSURE) for stat in ('mean', 'min')]
               + [f'{sensor}_mean_z_{suffix}' for sensor in (FLOW, PRESSURE) for suffix in ('r6h', 'chg6')],
    '누설': [f'{PRESSURE}_mean_z_r24h', f'{PRESSURE}_mean_z_r7d', f'{PRESSURE}_mean_z_chg24',
             'since_any', 'since_leak', 'n7d_any'],
}
coverage = features.loc[is_test, f'{PRESSURE}_mean_z'].notna().mean()
print({name: len(cols) for name, cols in DETECTOR_INPUTS.items()}, f'· 평가 구간 냉각 압력 z가 있는 행 {coverage:.1%}')

{'압력 저하': 16, '압력 과다': 8, '누설': 6} · 평가 구간 냉각 압력 z가 있는 행 97.6%


## 4. 경보·채점 함수

`field_eval.ipynb`와 같은 규칙이다. 경보 기준을 정하는 함수만 오경보 허용량을 인자로 받게 바꿨다.

In [3]:
def make_alarms(times, scores, threshold):
    # 점수가 기준을 넘는 시각들 중, 직전 경보 뒤 24시간이 지난 것만 경보
    over_times = times[scores > threshold]
    alarm_times, pos = [], 0
    while pos < len(over_times):
        alarm_times.append(over_times[pos])
        pos = np.searchsorted(over_times, over_times[pos] + ALARM_WINDOW, 'left')
    return np.asarray(alarm_times, dtype=float)

def count_false_alarms(alarm_times, fault_times):
    # 경보 뒤 24시간 안에 그 종류 고장이 없으면 오경보
    if len(alarm_times) == 0:
        return 0
    if len(fault_times) == 0:
        return len(alarm_times)
    next_pos = np.searchsorted(fault_times, alarm_times, 'right')
    next_fault = np.where(next_pos < len(fault_times), fault_times[np.minimum(next_pos, len(fault_times) - 1)], np.inf)
    return int(((next_fault - alarm_times) > ALARM_WINDOW).sum())

def split_events(fault_times):
    # 같은 종류 기록을 1일 간격으로 묶은 사건의 시작 시각, 반복 기록 시각
    if len(fault_times) == 0:
        return np.array([]), np.array([])
    is_new_event = np.r_[True, np.diff(fault_times) > DAY]
    return fault_times[is_new_event], fault_times[~is_new_event]

def calibrate_threshold(times, scores, fault_times, weeks, fa_target):
    # 보정 구간에서 오경보가 주 fa_target 이하가 되는 가장 낮은 점수 기준
    if len(scores) == 0 or weeks <= 0:
        return np.inf
    for quantile in np.r_[np.linspace(0.50, 0.99, 50), 1 - np.logspace(-2.2, -5, 15)]:
        threshold = np.quantile(scores, quantile)
        if count_false_alarms(make_alarms(times, scores, threshold), fault_times) / weeks <= fa_target:
            return threshold
    return np.inf

fault_times_by = {(tool, fault): faults.loc[(faults.Tool == tool) & (faults.fault_type == fault), 'time'].to_numpy()
                  for tool in tool_span.index for fault in FAULTS}
test_rows_by_tool = {tool: np.flatnonzero(is_test & (tool_of_row == tool)) for tool in tool_span.index}
calib_rows_by_tool = {tool: np.flatnonzero(is_calib & (tool_of_row == tool)) for tool in tool_span.index}
calib_weeks = ((tool_span.test_start - DAY - tool_span.fit_end) / WEEK).to_dict()

def evaluate(alarms_by_tool, fault):
    # 평가 구간 채점: 사건 탐지율, 판단 가능 사건 중 탐지율, 무작위 기대 탐지율, 오경보, 정밀도
    n_events = n_judgeable = n_detected = n_repeats = n_repeat_hits = n_alarms = n_false = 0
    lead_hours, random_hit_prob, total_weeks = [], [], 0.0
    for tool, span_row in tool_span.iterrows():
        alarm_times = np.sort(alarms_by_tool.get(tool, np.array([])))
        fault_times = fault_times_by[(tool, fault)]
        event_starts, repeat_times = split_events(fault_times)
        event_starts = event_starts[event_starts >= span_row.test_start + ALARM_WINDOW]
        repeat_times = repeat_times[repeat_times >= span_row.test_start + ALARM_WINDOW]
        test_seconds = span_row.t_last - span_row.test_start
        total_weeks += test_seconds / WEEK
        n_alarms += len(alarm_times)
        n_false += count_false_alarms(alarm_times, fault_times)
        test_starts = start_of_row[test_rows_by_tool[tool]]
        for event_start in event_starts:
            n_events += 1
            # 판단 가능: 사건 전 24시간 안에 셔터 1 기록이 하나라도 있음
            n_judgeable += int(np.searchsorted(test_starts, event_start - ALARM_WINDOW) < np.searchsorted(test_starts, event_start))
            hits = alarm_times[(alarm_times >= event_start - ALARM_WINDOW) & (alarm_times < event_start)]
            if len(hits):
                n_detected += 1
                lead_hours.append((event_start - hits[0]) / HOUR)
            random_hit_prob.append(1 - (1 - min(ALARM_WINDOW / test_seconds, 1)) ** len(alarm_times))
        for repeat_time in repeat_times:
            n_repeats += 1
            n_repeat_hits += int(((alarm_times >= repeat_time - ALARM_WINDOW) & (alarm_times < repeat_time)).any())
    return {'사건': n_events, '판단 가능 사건': n_judgeable, '탐지 사건': n_detected,
            '탐지율(%)': round(n_detected / max(n_events, 1) * 100, 1),
            '판단 가능 중 탐지율(%)': round(n_detected / max(n_judgeable, 1) * 100, 1),
            '무작위 기대 탐지율(%)': round(np.mean(random_hit_prob) * 100, 1) if random_hit_prob else np.nan,
            '선행 시간 중앙값(h)': round(np.median(lead_hours), 1) if lead_hours else np.nan,
            '경보 수': n_alarms, '오경보/주': round(n_false / total_weeks, 2),
            '정밀도(%)': round((n_alarms - n_false) / max(n_alarms, 1) * 100, 1),
            '반복 기록 탐지율(%)': round(n_repeat_hits / max(n_repeats, 1) * 100, 1)}

## 5. 학습과 평가

### 5-1. 탐지기 학습

- **지도 탐지기**(압력 저하·누설): 학습 구간에서, 기록 끝 24시간 안이라 정답을 알 수 없는 행은 빼고 학습한다. 결측(z 없음, 이동 평균 없음)은 0(= 평소)으로 채운다. `class_weight='balanced'`로 불균형을 맞춘다.
- **비지도 탐지기**(압력 과다): 학습 구간 **평소 행**(무작위 20만 개)으로만 학습한다. 정답은 쓰지 않는다. 점수 = `-score_samples` (클수록 평소와 다름).
- 비교용으로 압력 과다에도 **로지스틱 회귀(지도, 6시간 라벨)**를 같은 입력으로 학습한다. EDA 4-7c에서 유량이 약 6시간 전부터 떨어졌기 때문이다.

In [4]:
def fit_supervised(fault, input_cols, label_seconds):
    # 지도: 로지스틱 회귀, 정답 = label_seconds 안에 그 고장이 남
    ttf = segments[TTF[fault]].to_numpy()
    is_censored = ((segments.t_last.to_numpy() - segments.time_end.to_numpy()) < label_seconds) & np.isnan(ttf)
    label = (ttf <= label_seconds).astype(int)
    train_rows = np.flatnonzero(is_fit & ~is_censored)
    inputs = features[input_cols].fillna(0)
    model = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=2000))
    model.fit(inputs.iloc[train_rows], label[train_rows])
    return model.predict_proba(inputs)[:, 1], model

def fit_isolation(input_cols):
    # 비지도: 학습 구간 평소 행만으로 학습
    inputs = features[input_cols].fillna(0).to_numpy()
    normal_fit_rows = np.flatnonzero(is_fit & segments.is_normal.to_numpy())
    sample_rows = np.random.default_rng(SEED).choice(normal_fit_rows, min(200_000, len(normal_fit_rows)), replace=False)
    model = IsolationForest(n_estimators=200, max_samples=4096, random_state=SEED, n_jobs=-1).fit(inputs[sample_rows])
    return -model.score_samples(inputs), model

timer = time.time()
detectors = {}      # 고장 → (탐지기 이름, 점수, 모델)
drop_scores, drop_model = fit_supervised(FAULTS[0], DETECTOR_INPUTS['압력 저하'], DAY)
detectors[FAULTS[0]] = ('로지스틱 회귀 (지도, 24시간)', drop_scores, drop_model)
high_scores, high_model = fit_isolation(DETECTOR_INPUTS['압력 과다'])
detectors[FAULTS[1]] = ('IsolationForest (비지도)', high_scores, high_model)
leak_scores, leak_model = fit_supervised(FAULTS[2], DETECTOR_INPUTS['누설'], DAY)
detectors[FAULTS[2]] = ('로지스틱 회귀 (지도, 24시간)', leak_scores, leak_model)
high_lr_scores, high_lr_model = fit_supervised(FAULTS[1], DETECTOR_INPUTS['압력 과다'], 6 * HOUR)   # 비교용
print(f'학습 시간 {time.time() - timer:.0f}초')

학습 시간 8초


### 5-2. 규칙만 · 탐지기만 · 합친 경보 (오경보 허용 주 1회)

- **행 AUC**: 평가 구간에서 탐지기 점수로 "24시간 안에 그 고장" 행을 구분하는 정도(0.5 = 무작위). 경보 규칙과 무관하다.
- 합친 경보의 오경보는 규칙 경보의 오경보까지 포함한다.

In [5]:
def detector_alarms(scores, fault, fa_target):
    # 장비마다 보정 구간에서 기준을 정하고 평가 구간에 경보
    alarms_by_tool = {}
    for tool in tool_span.index:
        calib_rows, test_rows = calib_rows_by_tool[tool], test_rows_by_tool[tool]
        threshold = calibrate_threshold(start_of_row[calib_rows], scores[calib_rows], fault_times_by[(tool, fault)],
                                        calib_weeks[tool], fa_target)
        alarms_by_tool[tool] = make_alarms(start_of_row[test_rows], scores[test_rows], threshold)
    return alarms_by_tool

# ③ 직후 경보 규칙: 평가 구간에서 어떤 고장이든 기록되면 그 시각에 경보
rule_alarms = {}
for tool, span_row in tool_span.iterrows():
    any_fault_times = faults.loc[faults.Tool == tool, 'time'].to_numpy()
    any_fault_times = any_fault_times[any_fault_times >= span_row.test_start]
    rule_alarms[tool] = make_alarms(any_fault_times, np.ones(len(any_fault_times)), 0)

def combine_alarms(*alarm_dicts):
    # ④ 경보 합치기: 모든 경보 시각을 모은 뒤 24시간 재경보 금지를 다시 적용
    combined = {}
    for tool in tool_span.index:
        all_times = np.sort(np.concatenate([alarms.get(tool, np.array([])) for alarms in alarm_dicts]))
        combined[tool] = make_alarms(all_times, np.ones(len(all_times)), 0)
    return combined

def row_auc(scores, fault):
    label = segments[TTF[fault]].to_numpy()[is_test] <= DAY
    return round(roc_auc_score(label, scores[is_test]), 3) if 0 < label.sum() < len(label) else np.nan

result_rows = []
for fault in FAULTS:
    detector_name, scores, _ = detectors[fault]
    model_alarms = detector_alarms(scores, fault, FA_TARGET)
    result_rows.append({'고장': FSHORT[fault], '구성': '③ 규칙만', '행 AUC': np.nan, **evaluate(rule_alarms, fault)})
    result_rows.append({'고장': FSHORT[fault], '구성': f'② 탐지기만: {detector_name}', '행 AUC': row_auc(scores, fault),
                        **evaluate(model_alarms, fault)})
    result_rows.append({'고장': FSHORT[fault], '구성': '④ 합친 경보 (② + ③)', '행 AUC': np.nan,
                        **evaluate(combine_alarms(model_alarms, rule_alarms), fault)})
    if fault == FAULTS[1]:   # 압력 과다 비교용: 지도 6시간 라벨
        lr_alarms = detector_alarms(high_lr_scores, fault, FA_TARGET)
        result_rows.append({'고장': FSHORT[fault], '구성': '(비교) 로지스틱 회귀 (지도, 6시간) 만', '행 AUC': row_auc(high_lr_scores, fault),
                            **evaluate(lr_alarms, fault)})
        result_rows.append({'고장': FSHORT[fault], '구성': '(비교) 로지스틱 회귀 6시간 + 규칙', '행 AUC': np.nan,
                            **evaluate(combine_alarms(lr_alarms, rule_alarms), fault)})
results = pd.DataFrame(result_rows)
for fault in FAULTS:
    fault_rows = results[results.고장 == FSHORT[fault]]
    print(f'■ {FSHORT[fault]} — 사건 {fault_rows.사건.iloc[0]}건 (판단 가능 {fault_rows["판단 가능 사건"].iloc[0]}건)')
    display(fault_rows.drop(columns=['고장', '사건', '판단 가능 사건']).reset_index(drop=True))

■ 압력 저하 — 사건 67건 (판단 가능 67건)


,구성,행 AUC,탐지 사건,탐지율(%),판단 가능 중 탐지율(%),무작위 기대 탐지율(%),선행 시간 중앙값(h),경보 수,오경보/주,정밀도(%),반복 기록 탐지율(%)
0,③ 규칙만,NaN,4,6.0,6.0,12.7,7.3,154,0.36,25.3,78.5
1,"② 탐지기만: 로지스틱 회귀 (지도, 24시간)",0.712,21,31.3,31.3,26.3,9.4,539,1.59,6.9,52.3
2,④ 합친 경보 (② + ③),NaN,25,37.3,37.3,30.6,7.8,614,1.77,9.0,87.7


■ 압력 과다 — 사건 68건 (판단 가능 47건)


,구성,행 AUC,탐지 사건,탐지율(%),판단 가능 중 탐지율(%),무작위 기대 탐지율(%),선행 시간 중앙값(h),경보 수,오경보/주,정밀도(%),반복 기록 탐지율(%)
0,③ 규칙만,NaN,4,5.9,8.5,12.6,13.7,154,0.46,6.5,90.0
1,② 탐지기만: IsolationForest (비지도),0.712,22,32.4,46.8,24.2,14.1,541,1.64,4.3,40.0
2,④ 합친 경보 (② + ③),NaN,23,33.8,48.9,28.5,14.1,613,1.85,4.7,90.0
3,"(비교) 로지스틱 회귀 (지도, 6시간) 만",0.613,17,25.0,36.2,25.9,17.7,534,1.64,3.2,40.0
4,(비교) 로지스틱 회귀 6시간 + 규칙,NaN,18,26.5,38.3,30.7,17.7,640,1.96,3.4,80.0


■ 누설 — 사건 15건 (판단 가능 13건)


,구성,행 AUC,탐지 사건,탐지율(%),판단 가능 중 탐지율(%),무작위 기대 탐지율(%),선행 시간 중앙값(h),경보 수,오경보/주,정밀도(%),반복 기록 탐지율(%)
0,③ 규칙만,NaN,7,46.7,53.8,9.6,1.6,154,0.45,8.4,88.2
1,"② 탐지기만: 로지스틱 회귀 (지도, 24시간)",0.816,6,40.0,46.2,19.2,6.4,564,1.76,1.2,35.3
2,④ 합친 경보 (② + ③),NaN,9,60.0,69.2,23.3,1.6,668,2.07,2.1,88.2


### 5-3. 오경보 허용량별 탐지율

경보 기준을 정할 때의 오경보 허용량(주 0.25~5회)을 바꿔 가며, 탐지기만·합친 경보의 탐지율과 실제 오경보를 본다. 같은 경보 수를 무작위로 뿌렸을 때의 기대 탐지율도 함께 그린다.

In [6]:
FA_BUDGETS = [0.25, 0.5, 1, 2, 3, 5]
curve_rows = []
for fault in FAULTS:
    _, scores, _ = detectors[fault]
    for budget in FA_BUDGETS:
        model_alarms = detector_alarms(scores, fault, budget)
        for setup, alarms in [('탐지기만', model_alarms), ('합친 경보', combine_alarms(model_alarms, rule_alarms))]:
            curve_rows.append({'고장': FSHORT[fault], '구성': setup, '허용량(주)': budget, **evaluate(alarms, fault)})
curve = pd.DataFrame(curve_rows)
display(curve.pivot_table(index=['고장', '허용량(주)'], columns='구성',
                          values=['탐지율(%)', '판단 가능 중 탐지율(%)', '오경보/주', '정밀도(%)']).round(1))

# 그림 5-3: x = 실제 오경보/주, y = 사건 탐지율
fig = make_subplots(rows=1, cols=3, subplot_titles=[FSHORT[fault] for fault in FAULTS], shared_yaxes=True)
setup_colors = {'탐지기만': '#2a78d6', '합친 경보': '#eb6834'}
for col, fault in enumerate(FAULTS, 1):
    fault_curve = curve[curve.고장 == FSHORT[fault]]
    for setup, color in setup_colors.items():
        setup_curve = fault_curve[fault_curve.구성 == setup].sort_values('오경보/주')
        fig.add_scatter(x=setup_curve['오경보/주'], y=setup_curve['탐지율(%)'], mode='lines+markers', name=setup,
                        line=dict(color=color), legendgroup=setup, showlegend=col == 1, row=1, col=col)
    random_curve = fault_curve[fault_curve.구성 == '합친 경보'].sort_values('오경보/주')
    fig.add_scatter(x=random_curve['오경보/주'], y=random_curve['무작위 기대 탐지율(%)'], mode='lines', name='같은 경보 수로 무작위',
                    line=dict(color='#9aa3ab', dash='dot'), legendgroup='random', showlegend=col == 1, row=1, col=col)
    rule_result = results[(results.고장 == FSHORT[fault]) & (results.구성 == '③ 규칙만')].iloc[0]
    fig.add_scatter(x=[rule_result['오경보/주']], y=[rule_result['탐지율(%)']], mode='markers', name='규칙만',
                    marker=dict(symbol='diamond', size=11, color='#333'), legendgroup='rule', showlegend=col == 1, row=1, col=col)
fig.update_xaxes(title_text='실제 오경보/주 (장비 1대)')
fig.update_yaxes(title_text='사건 탐지율(%)', col=1)
fig.update_layout(title='그림 5-3. 오경보 허용량별 사건 탐지율 (평가 구간 = 장비별 뒤 25%)', legend=dict(orientation='h', y=-0.2))
show(fig, 520)

오경보/주       정밀도(%)       탐지율(%)       판단 가능 중 탐지율(%)      
구성            탐지기만 합친 경보   탐지기만 합친 경보   탐지기만 합친 경보           탐지기만 합친 경보
고장    허용량(주)                                                           
누설    0.25     0.9   1.3    0.7   3.1   13.3  53.3           15.4  61.5
      0.50     1.5   1.8    0.9   2.2   26.7  53.3           30.8  61.5
      1.00     1.8   2.1    1.2   2.1   40.0  60.0           46.2  69.2
      2.00     2.1   2.3    1.5   2.0   53.3  66.7           61.5  76.9
      3.00     2.6   2.8    1.3   1.8   60.0  66.7           69.2  76.9
      5.00     3.0   3.1    1.3   1.6   60.0  66.7           69.2  76.9
압력 과다 0.25     0.6   0.9    7.5   7.5   20.6  26.5           29.8  38.3
      0.50     1.0   1.3    5.4   6.2   26.5  32.4           38.3  46.8
      1.00     1.6   1.8    4.3   4.7   32.4  33.8           46.8  48.9
      2.00     2.8   3.0    3.6   3.6   42.6  41.2           61.7  59.6
      3.00     3.7   3.8    3.1   3.1   50.0  50.0           72.3  72.3
      5.00     5.2   5.2    2.5   2.5   57.4  58.8           83.0  85.1
압력 저하 0.25     0.8   1.1    8.7  12.8   22.4  26.9           22.4  26.9
      0.50     1.1   1.3    7.7  10.8   23.9  29.9           23.9  29.9
      1.00     1.6   1.8    6.9   9.0   31.3  37.3           31.3  37.3
      2.00     2.5   2.6    6.7   7.4   49.3  53.7           49.3  53.7
      3.00     2.9   3.0    6.2   6.5   55.2  59.7           55.2  59.7
      5.00     3.2   3.3    5.9   6.3   58.2  62.7           58.2  62.7

### 5-4. 지도 탐지기가 본 것 (로지스틱 회귀 계수)

입력을 표준화한 뒤의 계수다. **양수면 값이 클수록 고장 위험이 높다고 본 것**이다. 현장에서 경보 이유를 설명할 때 쓸 수 있다.

In [7]:
coef_tables = []
for fault, model in [(FAULTS[0], drop_model), (FAULTS[2], leak_model)]:
    logistic = model.named_steps['logisticregression']
    coef_tables.append(pd.Series(logistic.coef_[0], index=DETECTOR_INPUTS[FSHORT[fault]], name=FSHORT[fault]))
coef_table = pd.concat(coef_tables, axis=1)
display(coef_table.reindex(coef_table.abs().max(axis=1).sort_values(ascending=False).index).round(3))

,압력 저하,누설
since_drop,-1.271,NaN
since_leak,0.654,-0.759
n7d_any,0.746,0.396
since_high,0.358,NaN
FLOWCOOLPRESSURE_mean_z_r24h,0.023,-0.349
FLOWCOOLPRESSURE_mean_z_r7d,-0.007,-0.265
since_any,-0.259,0.105
FLOWCOOLPRESSURE_mean_z_chg24,-0.059,-0.243
FLOWCOOLFLOWRATE_mean_z_r24h,-0.224,NaN
FLOWCOOLPRESSURE_mean_z_r6h,0.158,NaN


### 5-5. 오경보 허용량 없이 — 모델 기본 판정으로 경보

5-2는 장비마다 "보정 구간 오경보 주 1회 이하"가 되도록 경보 기준을 정했다. 여기서는 그 조정 없이 **모델이 원래 내놓는 판정**으로 경보를 울린다. 경보 뒤 24시간 재경보 금지와 채점 방식은 같다.

| 탐지기 | 기본 판정 |
|---|---|
| 로지스틱 회귀 (지도) | 고장 확률 ≥ 0.5 (`predict`와 같음) |
| IsolationForest (비지도) | `predict`가 이상(−1)으로 판정 (`score_samples` < `offset_`) |
| (참고) 항상 경보 | 모든 행에서 경보 → 24시간마다 한 번. 탐지율의 상한과 그때의 오경보를 보여 준다 |

장비별 기준이 없으므로 모든 장비에 같은 기준이 적용된다.

In [8]:
def fixed_threshold_alarms(scores, threshold):
    # 장비 구분 없이 같은 기준으로 평가 구간에 경보
    return {tool: make_alarms(start_of_row[test_rows_by_tool[tool]], scores[test_rows_by_tool[tool]], threshold)
            for tool in tool_span.index}

default_thresholds = {FAULTS[0]: 0.5, FAULTS[1]: -high_model.offset_, FAULTS[2]: 0.5}   # IF 점수 = −score_samples
always_scores = np.ones(len(segments))

no_budget_rows = []
for fault in FAULTS:
    detector_name, scores, _ = detectors[fault]
    default_alarms = fixed_threshold_alarms(scores, default_thresholds[fault])
    always_alarms = fixed_threshold_alarms(always_scores, 0)
    is_alarm_row = scores[is_test] > default_thresholds[fault]
    for setup, alarms in [(f'② 탐지기만 (기본 판정): {detector_name}', default_alarms),
                          ('④ 합친 경보 (기본 판정 + 규칙)', combine_alarms(default_alarms, rule_alarms)),
                          ('(참고) 항상 경보', always_alarms)]:
        no_budget_rows.append({'고장': FSHORT[fault], '구성': setup,
                               '경보 판정 행(%)': round(is_alarm_row.mean() * 100, 1) if '항상' not in setup else 100.0,
                               **evaluate(alarms, fault)})
no_budget = pd.DataFrame(no_budget_rows)
for fault in FAULTS:
    fault_rows = no_budget[no_budget.고장 == FSHORT[fault]]
    print(f'■ {FSHORT[fault]} — 사건 {fault_rows.사건.iloc[0]}건 (판단 가능 {fault_rows["판단 가능 사건"].iloc[0]}건)')
    display(fault_rows.drop(columns=['고장', '사건', '판단 가능 사건']).reset_index(drop=True))

■ 압력 저하 — 사건 67건 (판단 가능 67건)


,구성,경보 판정 행(%),탐지 사건,탐지율(%),판단 가능 중 탐지율(%),무작위 기대 탐지율(%),선행 시간 중앙값(h),경보 수,오경보/주,정밀도(%),반복 기록 탐지율(%)
0,"② 탐지기만 (기본 판정): 로지스틱 회귀 (지도, 24시간)",26.5,29,43.3,43.3,25.8,9.4,590,1.70,8.8,73.8
1,④ 합친 경보 (기본 판정 + 규칙),26.5,32,47.8,47.8,29.7,8.0,652,1.86,9.8,96.9
2,(참고) 항상 경보,100.0,64,95.5,95.5,59.1,9.2,1949,5.88,4.7,96.9


■ 압력 과다 — 사건 68건 (판단 가능 47건)


,구성,경보 판정 행(%),탐지 사건,탐지율(%),판단 가능 중 탐지율(%),무작위 기대 탐지율(%),선행 시간 중앙값(h),경보 수,오경보/주,정밀도(%),반복 기록 탐지율(%)
0,② 탐지기만 (기본 판정): IsolationForest (비지도),38.0,33,48.5,70.2,44.7,10.8,1252,3.85,2.8,70.0
1,④ 합친 경보 (기본 판정 + 규칙),38.0,33,48.5,70.2,46.4,10.8,1287,3.96,2.9,90.0
2,(참고) 항상 경보,100.0,43,63.2,91.5,58.0,6.5,1949,6.03,2.3,80.0


■ 누설 — 사건 15건 (판단 가능 13건)


,구성,경보 판정 행(%),탐지 사건,탐지율(%),판단 가능 중 탐지율(%),무작위 기대 탐지율(%),선행 시간 중앙값(h),경보 수,오경보/주,정밀도(%),반복 기록 탐지율(%)
0,"② 탐지기만 (기본 판정): 로지스틱 회귀 (지도, 24시간)",43.7,10,66.7,76.9,56.1,7.7,824,2.57,1.6,82.4
1,④ 합친 경보 (기본 판정 + 규칙),43.7,10,66.7,76.9,56.3,8.1,875,2.72,1.7,94.1
2,(참고) 항상 경보,100.0,11,73.3,84.6,58.0,6.9,1949,6.13,0.7,88.2


### 5-6. 놓침 최소화 — 재현율 목표로 경보 기준 정하기

고장을 놓치는 것(미탐지)이 오경보보다 비싸다고 보고, 기준을 **오경보 허용량이 아니라 재현율 목표**로 정한다.

- **보정 구간에서** 합친 경보(탐지기 + 규칙)의 사건 탐지율이 목표(70·80·90%) 이상이 되는 가장 높은 기준을 고른다. 평가 구간은 채점에만 쓴다.
- 보정 구간 사건은 장비당 몇 건뿐이라 장비마다 따로 맞출 수 없다. 그래서 **모든 장비에 같은 분위수 q**를 쓰고, 기준 값은 장비마다 자기 보정 구간 점수의 q 분위수로 정한다(장비별 점수 수준 차이는 그대로 반영).
- 보정 구간 사건: 보정 구간 시작 1일 뒤 ~ 보정 구간 끝에 첫 기록이 있는 사건.

In [9]:
RECALL_TARGETS = [0.7, 0.8, 0.9]
QUANTILE_GRID = np.r_[1 - np.logspace(-4, -1, 25), np.linspace(0.89, 0.0, 90)]   # 높은 기준부터 낮춰 간다

def alarms_at_quantile(scores, quantile, row_lookup):
    # 장비마다 자기 보정 구간 점수의 quantile 분위수를 기준으로 row_lookup 구간에 경보
    alarms_by_tool = {}
    for tool in tool_span.index:
        calib_scores = scores[calib_rows_by_tool[tool]]
        threshold = np.quantile(calib_scores, quantile) if len(calib_scores) else np.inf
        target_rows = row_lookup[tool]
        alarms_by_tool[tool] = make_alarms(start_of_row[target_rows], scores[target_rows], threshold)
    return alarms_by_tool

# 보정 구간의 규칙 경보와 사건
calib_rule_alarms, calib_events = {}, {}
for tool, span_row in tool_span.iterrows():
    any_fault_times = faults.loc[faults.Tool == tool, 'time'].to_numpy()
    in_calib = (any_fault_times >= span_row.fit_end) & (any_fault_times < span_row.test_start - DAY)
    calib_rule_alarms[tool] = make_alarms(any_fault_times[in_calib], np.ones(in_calib.sum()), 0)
    for fault in FAULTS:
        event_starts, _ = split_events(fault_times_by[(tool, fault)])
        calib_events[(tool, fault)] = event_starts[(event_starts >= span_row.fit_end + DAY) & (event_starts < span_row.test_start - DAY)]

def calib_recall(alarms_by_tool, fault):
    # 보정 구간 사건 중 첫 기록 전 24시간 안에 경보가 있던 비율
    n_events = n_detected = 0
    for tool in tool_span.index:
        alarm_times = np.sort(alarms_by_tool[tool])
        for event_start in calib_events[(tool, fault)]:
            n_events += 1
            n_detected += int(((alarm_times >= event_start - ALARM_WINDOW) & (alarm_times < event_start)).any())
    return n_detected / max(n_events, 1), n_events

recall_rows = []
for fault in FAULTS:
    detector_name, scores, _ = detectors[fault]
    for target in RECALL_TARGETS:
        chosen_quantile, reached = None, np.nan
        for quantile in QUANTILE_GRID:
            calib_alarms = combine_alarms(alarms_at_quantile(scores, quantile, calib_rows_by_tool), calib_rule_alarms)
            reached, n_calib_events = calib_recall(calib_alarms, fault)
            if reached >= target:
                chosen_quantile = quantile
                break
        if chosen_quantile is None:
            chosen_quantile = QUANTILE_GRID[-1]
        test_alarms = combine_alarms(alarms_at_quantile(scores, chosen_quantile, test_rows_by_tool), rule_alarms)
        recall_rows.append({'고장': FSHORT[fault], '재현율 목표': f'{target:.0%}', '보정 사건': n_calib_events,
                            '보정 탐지율(%)': round(reached * 100, 1), '분위수 q': round(chosen_quantile, 4),
                            **evaluate(test_alarms, fault)})
recall_table = pd.DataFrame(recall_rows)
show_cols = ['고장', '재현율 목표', '보정 사건', '보정 탐지율(%)', '분위수 q', '사건', '탐지율(%)', '판단 가능 중 탐지율(%)',
             '무작위 기대 탐지율(%)', '선행 시간 중앙값(h)', '오경보/주', '정밀도(%)', '반복 기록 탐지율(%)']
display(recall_table[show_cols])

# 그림 5-6: 재현율 목표별 평가 구간 탐지율과 오경보
fig = make_subplots(rows=1, cols=2, subplot_titles=['평가 구간 사건 탐지율(%)', '평가 구간 오경보/주 (장비 1대)'])
fault_colors = dict(zip([FSHORT[fault] for fault in FAULTS], ['#2a78d6', '#eb6834', '#3a9e5f']))
for fault_name, color in fault_colors.items():
    fault_rows = recall_table[recall_table.고장 == fault_name]
    fig.add_scatter(x=fault_rows['재현율 목표'], y=fault_rows['탐지율(%)'], mode='lines+markers', name=fault_name,
                    line=dict(color=color), legendgroup=fault_name, row=1, col=1)
    fig.add_scatter(x=fault_rows['재현율 목표'], y=fault_rows['무작위 기대 탐지율(%)'], mode='lines', line=dict(color=color, dash='dot'),
                    name=f'{fault_name} 무작위', legendgroup=fault_name, showlegend=False, row=1, col=1)
    fig.add_scatter(x=fault_rows['재현율 목표'], y=fault_rows['오경보/주'], mode='lines+markers', line=dict(color=color),
                    legendgroup=fault_name, showlegend=False, row=1, col=2)
fig.update_xaxes(title_text='보정 구간 재현율 목표')
fig.update_layout(title='그림 5-6. 놓침 최소화 기준: 재현율 목표별 결과 (점선 = 같은 경보 수로 무작위)', legend=dict(orientation='h', y=-0.25))
show(fig, 460)

,고장,재현율 목표,보정 사건,보정 탐지율(%),분위수 q,사건,탐지율(%),판단 가능 중 탐지율(%),무작위 기대 탐지율(%),선행 시간 중앙값(h),오경보/주,정밀도(%),반복 기록 탐지율(%)
0,압력 저하,70%,109,71.6,0.21,67,82.1,82.1,50.7,9.0,4.45,5.6,96.9
1,압력 저하,80%,109,80.7,0.10,67,89.6,89.6,51.7,8.6,4.64,5.6,96.9
2,압력 저하,90%,109,87.2,0.00,67,94.0,94.0,59.0,9.1,5.70,4.8,96.9
3,압력 과다,70%,50,58.0,0.00,68,61.8,89.4,58.0,7.2,6.03,2.3,80.0
4,압력 과다,80%,50,58.0,0.00,68,61.8,89.4,58.0,7.2,6.03,2.3,80.0
5,압력 과다,90%,50,58.0,0.00,68,61.8,89.4,58.0,7.2,6.03,2.3,80.0
6,누설,70%,16,75.0,0.25,15,66.7,76.9,46.8,8.1,3.72,1.3,94.1
7,누설,80%,16,81.2,0.12,15,60.0,69.2,54.5,10.1,4.15,1.1,94.1
8,누설,90%,16,93.8,0.04,15,66.7,76.9,57.2,8.1,4.80,1.0,94.1


### 5-7. 베이스 모델과 비교

모델이 의미가 있으려면 학습 없는 간단한 방법보다 나아야 한다. 베이스 3개와 탐지기, 최종 구성(탐지기 + 규칙)을 같은 방식으로 비교한다.

| 구분 | 구성 | 학습 |
|---|---|---|
| 베이스 1 | **항상 경보**: 24시간마다 무조건 경보. 재현율 상한 | 없음 |
| 베이스 2 | **직후 경보 규칙**: 어떤 고장이든 나면 경보 | 없음 |
| 베이스 3 | **센서 하나 임계값**: 장비별 평소 대비 z값이 낮으면 경보. 압력 저하·누설 = 냉각 압력, 압력 과다 = 냉각 유량 (EDA 4-5, 4-7c에서 고장 전에 낮아진 센서) | 없음 (기준만 보정 구간에서 정함) |
| 베이스 3 + 규칙 | 센서 임계값 경보 + 직후 규칙 | 없음 |
| 탐지기 | 로지스틱 회귀 (지도) / IsolationForest (비지도) | 있음 |
| 최종 | 탐지기 + 직후 규칙 | 있음 |

기준은 두 가지로 정한다. **(A) 오경보 허용 주 1회** (5-2와 같음), **(B) 놓침 최소화, 보정 구간 재현율 80% 목표** (5-6과 같음). z값이 없는 행은 경보하지 않는다.

In [10]:
# 베이스 3 점수: 센서 z가 낮을수록 높은 점수 (z 없는 행은 경보하지 않도록 아주 낮게)
NO_SCORE = -1e9
single_sensor = {FAULTS[0]: f'{PRESSURE}_mean_z', FAULTS[1]: f'{FLOW}_mean_z', FAULTS[2]: f'{PRESSURE}_mean_z'}
single_scores = {fault: np.nan_to_num(-features[column].to_numpy(dtype=float), nan=NO_SCORE) for fault, column in single_sensor.items()}

def pick_recall_quantile(scores, fault, target):
    # 5-6과 같은 방법: 보정 구간에서 (점수 경보 + 규칙)의 사건 탐지율이 target 이상이 되는 가장 높은 분위수
    for quantile in QUANTILE_GRID:
        calib_alarms = combine_alarms(alarms_at_quantile(scores, quantile, calib_rows_by_tool), calib_rule_alarms)
        if calib_recall(calib_alarms, fault)[0] >= target:
            return quantile
    return QUANTILE_GRID[-1]

def row_auc_any(scores, fault):
    # z가 있는 행만으로 행 AUC
    label = segments[TTF[fault]].to_numpy()[is_test] <= DAY
    has_score = scores[is_test] > NO_SCORE
    return round(roc_auc_score(label[has_score], scores[is_test][has_score]), 3)

always_alarms = fixed_threshold_alarms(np.ones(len(segments)), 0)
baseline_rows = []
for fault in FAULTS:
    detector_name, detector_scores, _ = detectors[fault]
    sensor_scores = single_scores[fault]
    sensor_label = '냉각 유량' if fault == FAULTS[1] else '냉각 압력'
    # (A) 오경보 허용 주 1회
    sensor_alarms_a = detector_alarms(sensor_scores, fault, FA_TARGET)
    detector_alarms_a = detector_alarms(detector_scores, fault, FA_TARGET)
    # (B) 보정 구간 재현율 80% 목표
    sensor_quantile = pick_recall_quantile(sensor_scores, fault, 0.8)
    final_quantile = pick_recall_quantile(detector_scores, fault, 0.8)
    sensor_alarms_b = combine_alarms(alarms_at_quantile(sensor_scores, sensor_quantile, test_rows_by_tool), rule_alarms)
    final_alarms_b = combine_alarms(alarms_at_quantile(detector_scores, final_quantile, test_rows_by_tool), rule_alarms)
    setups = [
        ('베이스 1. 항상 경보', '-', np.nan, always_alarms),
        ('베이스 2. 직후 경보 규칙', '-', np.nan, rule_alarms),
        (f'베이스 3. {sensor_label} 임계값', 'A 주 1회', row_auc_any(sensor_scores, fault), sensor_alarms_a),
        ('베이스 3 + 규칙', 'A 주 1회', np.nan, combine_alarms(sensor_alarms_a, rule_alarms)),
        (f'탐지기: {detector_name}', 'A 주 1회', row_auc(detector_scores, fault), detector_alarms_a),
        ('최종: 탐지기 + 규칙', 'A 주 1회', np.nan, combine_alarms(detector_alarms_a, rule_alarms)),
        ('베이스 3 + 규칙', 'B 재현율 80%', np.nan, sensor_alarms_b),
        ('최종: 탐지기 + 규칙', 'B 재현율 80%', np.nan, final_alarms_b),
    ]
    for setup, criterion, auc_value, alarms in setups:
        baseline_rows.append({'고장': FSHORT[fault], '구성': setup, '기준': criterion, '행 AUC': auc_value, **evaluate(alarms, fault)})
baseline_table = pd.DataFrame(baseline_rows)
compare_cols = ['구성', '기준', '행 AUC', '탐지율(%)', '판단 가능 중 탐지율(%)', '경보 수', '오경보/주', '정밀도(%)', '반복 기록 탐지율(%)']
for fault in FAULTS:
    fault_rows = baseline_table[baseline_table.고장 == FSHORT[fault]]
    print(f'■ {FSHORT[fault]} — 사건 {fault_rows.사건.iloc[0]}건 (판단 가능 {fault_rows["판단 가능 사건"].iloc[0]}건)')
    display(fault_rows[compare_cols].reset_index(drop=True))

# 그림 5-7: 오경보/주 vs 탐지율, 베이스와 최종
fig = make_subplots(rows=1, cols=3, subplot_titles=[FSHORT[fault] for fault in FAULTS], shared_yaxes=True)
setup_styles = {'베이스 1': ('#9aa3ab', 'square'), '베이스 2': ('#333333', 'diamond'), '베이스 3': ('#2a78d6', 'circle'),
                '탐지기': ('#7b5cc4', 'triangle-up'), '최종': ('#eb6834', 'star')}
for col, fault in enumerate(FAULTS, 1):
    fault_rows = baseline_table[baseline_table.고장 == FSHORT[fault]]
    for _, row in fault_rows.iterrows():
        style_key = next(key for key in setup_styles if row.구성.startswith(key))
        color, symbol = setup_styles[style_key]
        short_name = row.구성.split(':')[0]
        label = f'{short_name} ({row.기준})' if row.기준 != '-' else row.구성
        is_recall_criterion = row.기준.startswith('B')
        fig.add_scatter(x=[row['오경보/주']], y=[row['탐지율(%)']], mode='markers', name=label, legendgroup=label,
                        marker=dict(color=color, symbol=symbol, size=13 if is_recall_criterion else 10,
                                    line=dict(width=1.5 if is_recall_criterion else 0, color='#14213D')),
                        showlegend=col == 1, row=1, col=col)
fig.update_xaxes(title_text='실제 오경보/주 (장비 1대)')
fig.update_yaxes(title_text='사건 탐지율(%)', col=1)
fig.update_layout(title='그림 5-7. 베이스 모델과 최종 구성 비교 (테두리 = 재현율 80% 목표 기준)', legend=dict(orientation='h', y=-0.25))
show(fig, 560)

■ 압력 저하 — 사건 67건 (판단 가능 67건)


,구성,기준,행 AUC,탐지율(%),판단 가능 중 탐지율(%),경보 수,오경보/주,정밀도(%),반복 기록 탐지율(%)
0,베이스 1. 항상 경보,-,NaN,95.5,95.5,1949,5.88,4.7,96.9
1,베이스 2. 직후 경보 규칙,-,NaN,6.0,6.0,154,0.36,25.3,78.5
2,베이스 3. 냉각 압력 임계값,A 주 1회,0.518,29.9,29.9,629,1.87,6.2,58.5
3,베이스 3 + 규칙,A 주 1회,NaN,32.8,32.8,716,2.10,7.4,80.0
4,"탐지기: 로지스틱 회귀 (지도, 24시간)",A 주 1회,0.712,31.3,31.3,539,1.59,6.9,52.3
5,최종: 탐지기 + 규칙,A 주 1회,NaN,37.3,37.3,614,1.77,9.0,87.7
6,베이스 3 + 규칙,B 재현율 80%,NaN,88.1,88.1,1767,5.32,4.9,95.4
7,최종: 탐지기 + 규칙,B 재현율 80%,NaN,89.6,89.6,1553,4.64,5.6,96.9


■ 압력 과다 — 사건 68건 (판단 가능 47건)


,구성,기준,행 AUC,탐지율(%),판단 가능 중 탐지율(%),경보 수,오경보/주,정밀도(%),반복 기록 탐지율(%)
0,베이스 1. 항상 경보,-,NaN,63.2,91.5,1949,6.03,2.3,80.0
1,베이스 2. 직후 경보 규칙,-,NaN,5.9,8.5,154,0.46,6.5,90.0
2,베이스 3. 냉각 유량 임계값,A 주 1회,0.799,23.5,34.0,270,0.80,5.9,10.0
3,베이스 3 + 규칙,A 주 1회,NaN,26.5,38.3,384,1.14,6.0,80.0
4,탐지기: IsolationForest (비지도),A 주 1회,0.712,32.4,46.8,541,1.64,4.3,40.0
5,최종: 탐지기 + 규칙,A 주 1회,NaN,33.8,48.9,613,1.85,4.7,90.0
6,베이스 3 + 규칙,B 재현율 80%,NaN,60.3,87.2,1945,6.02,2.3,80.0
7,최종: 탐지기 + 규칙,B 재현율 80%,NaN,61.8,89.4,1948,6.03,2.3,80.0


■ 누설 — 사건 15건 (판단 가능 13건)


,구성,기준,행 AUC,탐지율(%),판단 가능 중 탐지율(%),경보 수,오경보/주,정밀도(%),반복 기록 탐지율(%)
0,베이스 1. 항상 경보,-,NaN,73.3,84.6,1949,6.13,0.7,88.2
1,베이스 2. 직후 경보 규칙,-,NaN,46.7,53.8,154,0.45,8.4,88.2
2,베이스 3. 냉각 압력 임계값,A 주 1회,0.537,20.0,23.1,605,1.91,0.5,0.0
3,베이스 3 + 규칙,A 주 1회,NaN,53.3,61.5,705,2.19,1.8,58.8
4,"탐지기: 로지스틱 회귀 (지도, 24시간)",A 주 1회,0.816,40.0,46.2,564,1.76,1.2,35.3
5,최종: 탐지기 + 규칙,A 주 1회,NaN,60.0,69.2,668,2.07,2.1,88.2
6,베이스 3 + 규칙,B 재현율 80%,NaN,66.7,76.9,1824,5.73,0.8,88.2
7,최종: 탐지기 + 규칙,B 재현율 80%,NaN,60.0,69.2,1326,4.15,1.1,94.1


### 5-8. 분류 지표 — 정확도·재현율·정밀도·F1

두 단위로 적는다.

- **행 단위**: 평가 구간의 스텝 구간 하나하나를 "24시간 안에 그 고장(1) / 아님(0)"으로 맞혔는지. 탐지기 점수가 장비별 기준을 넘으면 1로 예측한다(재경보 금지·규칙은 적용하지 않음). 기록 끝 24시간 안이라 정답을 알 수 없는 행은 뺀다.
- **사건 단위**: 지금까지 쓴 채점. 재현율 = 사건 탐지율, 정밀도 = 맞은 경보 비율, F1 = 둘의 조화 평균. "고장 아님" 사건이 따로 없어서 **정확도는 계산할 수 없다**.

기준은 (A) 오경보 허용 주 1회, (B) 보정 구간 재현율 80% 목표(5-6과 같음)다.

In [11]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

def row_thresholds_fa(scores, fault, fa_target):
    # (A) 장비별 기준: 보정 구간 오경보 주 fa_target 이하
    return {tool: calibrate_threshold(start_of_row[calib_rows_by_tool[tool]], scores[calib_rows_by_tool[tool]],
                                      fault_times_by[(tool, fault)], calib_weeks[tool], fa_target) for tool in tool_span.index}

def row_thresholds_quantile(scores, quantile):
    # (B) 장비별 기준: 자기 보정 구간 점수의 quantile 분위수
    return {tool: np.quantile(scores[calib_rows_by_tool[tool]], quantile) if len(calib_rows_by_tool[tool]) else np.inf
            for tool in tool_span.index}

def row_metrics(scores, fault, thresholds):
    # 평가 구간 행 단위 분류 지표
    ttf = segments[TTF[fault]].to_numpy()
    is_censored = ((segments.t_last.to_numpy() - segments.time_end.to_numpy()) < DAY) & np.isnan(ttf)
    label_list, predicted_list = [], []
    for tool in tool_span.index:
        rows = test_rows_by_tool[tool]
        rows = rows[~is_censored[rows]]
        label_list.append(ttf[rows] <= DAY)
        predicted_list.append(scores[rows] > thresholds[tool])
    label, predicted = np.concatenate(label_list), np.concatenate(predicted_list)
    return {'정확도': accuracy_score(label, predicted), '재현율': recall_score(label, predicted, zero_division=0),
            '정밀도': precision_score(label, predicted, zero_division=0), 'F1': f1_score(label, predicted, zero_division=0),
            '양성 비율': label.mean()}

def event_metrics(result_row):
    # 사건 단위: 재현율 = 탐지율, 정밀도 = 맞은 경보 비율
    recall, precision = result_row['탐지율(%)'] / 100, result_row['정밀도(%)'] / 100
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return {'정확도': np.nan, '재현율': recall, '정밀도': precision, 'F1': f1}

metric_rows = []
for fault in FAULTS:
    detector_name, scores, _ = detectors[fault]
    final_quantile = pick_recall_quantile(scores, fault, 0.8)
    for criterion, thresholds in [('A 주 1회', row_thresholds_fa(scores, fault, FA_TARGET)),
                                  ('B 재현율 80%', row_thresholds_quantile(scores, final_quantile))]:
        metric_rows.append({'고장': FSHORT[fault], '단위': '행 (탐지기만)', '기준': criterion, **row_metrics(scores, fault, thresholds)})
    for criterion in ['A 주 1회', 'B 재현율 80%']:
        final_row = baseline_table[(baseline_table.고장 == FSHORT[fault]) & (baseline_table.구성 == '최종: 탐지기 + 규칙')
                                   & (baseline_table.기준 == criterion)].iloc[0]
        metric_rows.append({'고장': FSHORT[fault], '단위': '사건 (탐지기 + 규칙)', '기준': criterion, **event_metrics(final_row)})
metric_table = pd.DataFrame(metric_rows)
display(metric_table.round(2))

,고장,단위,기준,정확도,재현율,정밀도,F1,양성 비율
0,압력 저하,행 (탐지기만),A 주 1회,0.77,0.36,0.06,0.10,0.04
1,압력 저하,행 (탐지기만),B 재현율 80%,0.29,0.90,0.04,0.08,0.04
2,압력 저하,사건 (탐지기 + 규칙),A 주 1회,NaN,0.37,0.09,0.15,NaN
3,압력 저하,사건 (탐지기 + 규칙),B 재현율 80%,NaN,0.90,0.06,0.11,NaN
4,압력 과다,행 (탐지기만),A 주 1회,0.90,0.50,0.10,0.17,0.02
5,압력 과다,행 (탐지기만),B 재현율 80%,0.02,1.00,0.02,0.04,0.02
6,압력 과다,사건 (탐지기 + 규칙),A 주 1회,NaN,0.34,0.05,0.08,NaN
7,압력 과다,사건 (탐지기 + 규칙),B 재현율 80%,NaN,0.62,0.02,0.04,NaN
8,누설,행 (탐지기만),A 주 1회,0.71,0.46,0.01,0.02,0.01
9,누설,행 (탐지기만),B 재현율 80%,0.35,0.92,0.01,0.02,0.01


### 5-9. 운영 기준 — 재현율 우선, 정밀도 하한 (항상 경보의 2배)

이온 밀은 웨이퍼 불량과 비계획 정지 비용이 커서 **놓침(FN)이 더 비싸다.** 다만 경보마다 챔버를 열어 점검하는 비용도 있다. 그래서 **재현율을 우선하되, 정밀도가 일정 수준 아래로 내려가지 않게** 경보 기준을 잡는다.

- **정밀도 하한 = 항상 경보 정밀도 × 2.** 고장 종류마다 기본 발생률이 달라 고정 % 대신 "아무 정보 없이 매일 경보할 때보다 2배는 정확해야 한다"로 정했다.
- 항상 경보 정밀도(기본 비율)를 잴 구간을 두 가지로 비교한다. **가. 보정 구간**, **나. 학습+보정 구간**. 둘 다 평가 구간 정보는 쓰지 않는다. 나는 기간이 3배 길어 고장이 몰린 시기의 영향이 덜하다.
- **보정 구간에서** 기준을 낮은 쪽(경보 많음 = 재현율 높음)부터 올려 가며, 합친 경보(탐지기 + 규칙)의 정밀도가 하한 이상이 되는 **첫 기준**을 고른다. 이것이 하한을 지키는 기준 중 재현율이 가장 높은 기준이다.
- 분위수 q는 5-6처럼 모든 장비에 같게 쓰고, 기준 값은 장비마다 자기 보정 구간 점수의 q 분위수다.
- 하한을 지키는 기준이 없으면 보정 정밀도가 가장 높은 기준을 쓰고 '하한 못 맞춤'으로 표시한다.
- 정밀도·재현율은 **사건 단위**다(정밀도 = 경보 뒤 24시간 안에 그 종류 고장이 있던 경보 비율).

In [12]:
PRECISION_MULTIPLIER = 2.0                           # 정밀도 하한 = 항상 경보 정밀도 × 2
ASCENDING_QUANTILES = np.sort(QUANTILE_GRID)          # 낮은 기준(경보 많음)부터

def calib_precision(alarms_by_tool, fault):
    # 경보 중 24시간 안에 그 종류 고장이 온 비율 (사건 단위 정밀도)
    n_alarms = n_false = 0
    for tool in tool_span.index:
        alarm_times = np.sort(alarms_by_tool.get(tool, np.array([])))
        n_alarms += len(alarm_times)
        n_false += count_false_alarms(alarm_times, fault_times_by[(tool, fault)])
    return (n_alarms - n_false) / n_alarms if n_alarms else 0.0, n_alarms

def always_alarms_on(row_lookup):
    # 항상 경보: 셔터 1 기록이 있으면 24시간마다 1번
    return {tool: make_alarms(start_of_row[row_lookup[tool]], np.ones(len(row_lookup[tool])), 0) for tool in tool_span.index}

# 하한의 기본 비율을 잴 구간 두 가지 (둘 다 평가 구간 정보는 쓰지 않음)
fit_calib_rows_by_tool = {tool: np.flatnonzero((is_fit | is_calib) & (tool_of_row == tool)) for tool in tool_span.index}
FLOOR_BASES = {'가. 보정 구간': always_alarms_on(calib_rows_by_tool),
               '나. 학습+보정 구간': always_alarms_on(fit_calib_rows_by_tool)}

def pick_floor_quantile(scores, fault, precision_floor):
    # 보정 구간에서 낮은 기준부터 올려 가며 정밀도가 하한 이상인 첫 분위수 (없으면 정밀도 최고 분위수)
    best_quantile, best_precision = ASCENDING_QUANTILES[-1], -1.0
    for quantile in ASCENDING_QUANTILES:
        calib_alarms = combine_alarms(alarms_at_quantile(scores, quantile, calib_rows_by_tool), calib_rule_alarms)
        precision_value, n_calib_alarms = calib_precision(calib_alarms, fault)
        if n_calib_alarms and precision_value > best_precision:
            best_quantile, best_precision = quantile, precision_value
        if n_calib_alarms and precision_value >= precision_floor:
            return quantile, True
    return best_quantile, False

floor_rows = []
for fault in FAULTS:
    detector_name, scores, _ = detectors[fault]
    test_always_precision = evaluate(always_alarms, fault)['정밀도(%)']
    for base_name, base_alarms in FLOOR_BASES.items():
        base_precision, _ = calib_precision(base_alarms, fault)
        precision_floor = base_precision * PRECISION_MULTIPLIER
        chosen_quantile, reached_floor = pick_floor_quantile(scores, fault, precision_floor)
        calib_alarms = combine_alarms(alarms_at_quantile(scores, chosen_quantile, calib_rows_by_tool), calib_rule_alarms)
        calib_precision_value, _ = calib_precision(calib_alarms, fault)
        calib_recall_value, n_calib_events = calib_recall(calib_alarms, fault)
        test_result = evaluate(combine_alarms(alarms_at_quantile(scores, chosen_quantile, test_rows_by_tool), rule_alarms), fault)
        floor_rows.append({'고장': FSHORT[fault], '하한 기준 구간': base_name,
                           '항상 경보 정밀도(%)': round(base_precision * 100, 1),
                           '정밀도 하한(%)': round(precision_floor * 100, 1),
                           '하한 맞춤': '예' if reached_floor else '못 맞춤',
                           '분위수 q': round(chosen_quantile, 4),
                           '보정 사건': n_calib_events,
                           '보정 재현율(%)': round(calib_recall_value * 100, 1),
                           '보정 정밀도(%)': round(calib_precision_value * 100, 1),
                           '평가 사건': test_result['사건'],
                           '평가 재현율(%)': test_result['탐지율(%)'],
                           '판단 가능 중 재현율(%)': test_result['판단 가능 중 탐지율(%)'],
                           '평가 정밀도(%)': test_result['정밀도(%)'],
                           '평가 항상 경보 정밀도(%)': test_always_precision,
                           '오경보/주': test_result['오경보/주'],
                           '선행 시간 중앙값(h)': test_result['선행 시간 중앙값(h)']})
floor_table = pd.DataFrame(floor_rows)
display(floor_table)

,고장,하한 기준 구간,항상 경보 정밀도(%),정밀도 하한(%),하한 맞춤,분위수 q,보정 사건,보정 재현율(%),보정 정밀도(%),평가 사건,평가 재현율(%),판단 가능 중 재현율(%),평가 정밀도(%),평가 항상 경보 정밀도(%),오경보/주,선행 시간 중앙값(h)
0,압력 저하,가. 보정 구간,7.4,14.8,예,0.8800,109,16.5,15.0,67,35.8,35.8,8.7,4.7,1.86,8.0
1,압력 저하,나. 학습+보정 구간,6.4,12.8,예,0.8500,109,15.6,13.3,67,38.8,38.8,8.0,4.7,2.03,7.8
2,압력 과다,가. 보정 구간,1.8,3.6,예,0.9822,50,24.0,3.6,68,44.1,63.8,4.2,2.3,2.58,11.7
3,압력 과다,나. 학습+보정 구간,1.6,3.2,예,0.9822,50,24.0,3.6,68,44.1,63.8,4.2,2.3,2.58,11.7
4,누설,가. 보정 구간,0.8,1.5,예,0.7400,16,68.8,1.6,15,66.7,76.9,1.9,0.7,2.41,8.1
5,누설,나. 학습+보정 구간,0.8,1.6,예,0.7500,16,68.8,1.6,15,66.7,76.9,2.0,0.7,2.38,8.1


**5-9 해석** (모두 탐지기 + 직후 경보 규칙. 압력 저하·누설 = 로지스틱 회귀, **지도**, 정답 = 24시간 안 고장 / 압력 과다 = IsolationForest, **비지도**, 정답 없이 학습)

| 고장 | 하한 기준 구간 | 정밀도 하한 | 평가 재현율 | 평가 정밀도 (항상 경보 대비) | 오경보/주 | 선행 시간 중앙값 |
|---|---|---|---|---|---|---|
| 압력 저하 | 가. 보정 | 14.8% | 35.8% | 8.7% (1.9배) | 1.86 | 8.0시간 |
| 압력 저하 | **나. 학습+보정** | 12.8% | **38.8%** | 8.0% (1.7배) | 2.03 | 7.8시간 |
| 압력 과다 | 가 / 나 | 3.6% / 3.2% | **44.1%** (판단 가능 중 63.8%) | 4.2% (1.8배) | 2.58 | 11.7시간 |
| 누설 | 가 / 나 | 1.5% / 1.6% | **66.7%** | 1.9~2.0% (2.7배) | 2.38~2.41 | 8.1시간 |

- **세 고장 모두 두 방식 다 보정 구간에서 하한을 맞췄고**, 평가 구간에서도 정밀도가 항상 경보의 1.7~2.7배로 유지됐다.
- **하한 기준 구간을 바꿔도 차이는 압력 저하에서만, 작게 난다.** 학습+보정 구간의 기본 비율(6.4%)이 보정 구간(7.4%)보다 낮아 하한이 14.8% → 12.8%로 풀렸고, 평가 재현율이 35.8% → 38.8%(사건 67건 중 2건)로 올랐다. 대신 오경보가 주 0.17회 늘고 정밀도는 8.7% → 8.0%다. 압력 과다·누설은 같은 기준이 골라져 결과가 같다.
- **기본안은 나(학습+보정 구간)로 한다.** 기간이 3배 길어 고장이 몰린 시기의 영향이 덜하고, 놓침을 줄이는 쪽이라 현장 조건(FN이 더 비쌈)에 맞는다. 2건 차이는 우연일 수 있어 둘 사이의 우열은 확실하지 않다.
- 오경보 주 1회 기준(5-2)과 비교: 압력 저하 37.3% → 38.8%, 압력 과다 33.8% → 44.1%, 누설 60.0% → 66.7%. 이 기준은 **오경보 주 약 2~2.6회**에서 운영하는 것과 비슷하다(장비 1대당 2~3일에 한 번 점검 경보).
- 한계: 보정 사건이 누설 16건, 압력 과다 50건으로 적어 분위수 q가 흔들릴 수 있다. 재경보 금지(24시간) 때문에 기준을 낮춰도 보정 재현율이 꼭 오르지는 않는다(압력 저하 16.5% → 15.6%).

## 6. 해석과 한계

**결론: 고장 종류별 탐지기 + 직후 경보 규칙을 합치면 세 고장 모두 단일 모델보다 낫다.** 다만 정밀도는 여전히 10% 미만이고, 사건 수가 적어 차이 대부분은 우연 범위 안이다.

| 고장 (평가 사건) | 구성 (오경보 허용 주 1회) | 탐지율 | 판단 가능 중 | 같은 경보 수로 무작위 | 오경보/주 | 정밀도 | `field_eval` 최고 |
|---|---|---|---|---|---|---|---|
| 압력 저하 (67건) | 로지스틱 회귀(지도) + 규칙 | **37.3%** | 37.3% | 30.6% | 1.77 | 9.0% | 35.8% |
| 압력 과다 (68건) | IsolationForest(비지도) + 규칙 | **33.8%** | **48.9%** | 28.5% | 1.85 | 4.7% | 35.3% |
| 누설 (15건) | 로지스틱 회귀(지도) + 규칙 | **60.0%** | 69.2% | 23.3% | 2.07 | 2.1% | 46.7% |

**1. 규칙을 합치면 반복 고장을 거의 다 잡는다.** 반복 기록 탐지율이 탐지기만일 때 35~52%에서 합친 경보 88~90%로 오른다. 첫 고장 탐지율도 압력 저하 +6%p, 누설 +20%p 올랐다. 늘어난 오경보는 주 0.2~0.3회다.

**2. 압력 과다는 이상 감지(비지도)가 맞다.** 같은 입력으로 6시간 라벨 지도 모델을 학습하면 탐지율 25.0%(행 AUC 0.613)로, IsolationForest 32.4%(0.712)보다 낮다. EDA 4-7c(유량이 서서히 떨어짐)와 `field_eval`(지도 모델이 기간이 바뀌면 뒤집힘)과 같은 결론이다. 판단 가능한 사건만 보면 약 절반(48.9%)을 잡는다.

**3. 오경보를 조금 더 허용하면 압력 저하가 크게 오른다.** 허용량을 주 2회로 늘리면 합친 경보 탐지율이 압력 저하 53.7%(무작위 37.5%, +16%p), 압력 과다 41.2%(36.9%), 누설 66.7%(26.8%)다. 압력 저하는 허용량을 늘릴수록 무작위와의 차이가 커진다(주 1회 +7%p → 주 3회 +18%p). 압력 과다는 허용량을 늘려도 무작위와의 차이가 4%p 정도로 작다.

**4. 압력 저하 탐지기는 센서보다 고장 이력에 기댄다.** 계수가 큰 입력은 직전 압력 저하 이후 시간(−1.27, 최근일수록 위험), 최근 7일 고장 수(+0.75), 직전 누설 이후 시간이다. 냉각 압력 수준·변화 계수는 0.2 이하다. 장비 자신의 평소 기준으로 봐도 냉각 압력 신호가 약하다는 EDA·`self_baseline` 결과와 같다. **누설**은 지난 24시간·7일 냉각 압력이 낮을수록 위험(−0.35, −0.27)으로, EDA 4-7c(누설 전 압력이 낮음)와 방향이 맞는다.

**5. 오경보는 목표보다 많다.** 보정 구간에서 주 1회로 맞췄지만 평가 구간에서는 탐지기만 주 1.6~1.8회다. 시간이 지나며 장비 상태가 바뀌기 때문이다(EDA 부록: 압력 수준이 몇 달 단위로 이동). 기준을 주기적으로 다시 정하는 재학습이 필요하다.

**6. 오경보 기준을 없애면 (5-5, 모델 기본 판정)** 탐지율은 압력 저하 47.8%, 압력 과다 48.5%(판단 가능 중 70.2%), 누설 66.7%로 오르지만 오경보도 주 1.9~4.0회로 늘어난다. 모든 장비에 같은 기준이라 장비별 차이도 반영되지 않는다.

**7. 놓침 최소화 (5-6, 재현율 목표로 기준)**

| 고장 | 설정 | 평가 탐지율 | 판단 가능 중 | 오경보/주 | 정밀도 | 참고: 항상 경보 |
|---|---|---|---|---|---|---|
| 압력 저하 | 목표 80% | **89.6%** | 89.6% | 4.64 | 5.6% | 95.5%, 주 5.88회 |
| 압력 과다 | 목표 70% 이상 (보정 구간에서 58%까지만 도달) | 61.8% | **89.4%** | 6.03 | 2.3% | 63.2%, 주 6.03회 |
| 누설 | 목표 70~90% | 60~67% | 69~77% | 3.7~4.8 | 1.0~1.3% | 73.3%, 주 6.13회 |

- **놓침을 줄이려면 거의 매일 경보를 울려야 한다.** 압력 저하는 사건 10건 중 9건을 잡지만, 장비 한 대당 주 4~5회 오경보가 나고 경보 18번 중 1번만 맞다.
- **비교 기준은 "항상 경보"다.** 경보가 이렇게 많으면 무작위 기대 탐지율(근사)은 실제보다 낮게 나온다(항상 경보 95.5%인데 근사는 59.1%). 압력 저하 목표 80%는 항상 경보보다 오경보를 약 20% 줄이면서 탐지율은 6%p만 낮다. 신호가 주는 이득은 이 정도다.
- **압력 과다는 놓침의 대부분이 기록 없음 때문이다.** 68건 중 21건은 고장 전 24시간에 셔터 1 기록이 없다. 판단 가능한 사건으로는 89.4%를 잡는다. 보정 구간에서도 같은 이유로 목표 70%에 닿지 못해 사실상 항상 경보가 됐다.
- **누설은 사건이 15건이라** 목표를 올려도 평가 탐지율이 60~67%에서 흔들린다. 결과를 믿기 어렵다.
- **현장 적용**: 놓침을 줄이는 쪽이라면 압력 저하는 "재현율 80% 목표"(평가 89.6%, 주 4.6회), 압력 과다는 IsolationForest 기본 판정(판단 가능 중 70.2%, 주 3.9회)이 현실적인 출발점이다. 오경보를 감당할 수 있는지는 현장 비용(점검 1회 vs 고장 1회)으로 정해야 한다.

**8. 베이스 모델과 비교 (5-7)**

| 고장 | 베이스 3 + 규칙 (주 1회) | 최종 (주 1회) | 베이스 3 + 규칙 (재현율 80%) | 최종 (재현율 80%) |
|---|---|---|---|---|
| 압력 저하 | 32.8%, 주 2.10회 | **37.3%, 주 1.77회** | 88.1%, 주 5.32회 | **89.6%, 주 4.64회** |
| 압력 과다 | 26.5%, 주 1.14회 | 33.8%, 주 1.85회 | 60.3%, 주 6.02회 | 61.8%, 주 6.03회 |
| 누설 | 53.3%, 주 2.19회 | 60.0%, 주 2.07회 | 66.7%, 주 5.73회 | 60.0%, 주 4.15회 |

- **압력 저하는 최종 구성이 베이스보다 확실히 낫다.** 같은 기준에서 탐지율은 높고 오경보는 적다(주 1회 기준 +4.5%p·오경보 −0.33회, 재현율 80% 기준 +1.5%p·오경보 −0.68회). 냉각 압력 하나의 행 AUC는 0.518로 거의 무작위라, 차이는 주로 **고장 이력**에서 온다(5-4 계수와 같음).
- **압력 과다는 냉각 유량 하나로도 IsolationForest와 비슷하다.** 유량 임계값의 행 AUC(0.799)가 IsolationForest(0.712)보다 높다. 경보 기준에서는 탐지율이 낮지만(23.5% vs 32.4%) 오경보도 절반이다(주 0.80 vs 1.64회). 같은 오경보 수준(5-3, 주 약 1회에서 26.5%)으로 맞추면 차이는 3%p 정도다. **현장에서 설명하기 쉬운 "냉각 유량이 평소보다 낮으면 경보" 규칙이 출발점으로 충분하다.**
- **누설은 차이를 판단할 수 없다.** 사건 15건이라 1건이 6.7%p다. 최종과 베이스의 차이는 1~2건이다.
- **항상 경보(베이스 1)**는 재현율 64~96%의 상한이지만 오경보가 주 6회다. 재현율 80% 목표의 최종 구성은 압력 저하에서 항상 경보보다 오경보를 21% 줄인다. 압력 과다는 보정 구간에서 목표에 닿지 못해 사실상 항상 경보와 같다.

**9. 운영 기준 (5-9): 재현율 우선 + 정밀도 하한(항상 경보의 2배, 기본 비율은 학습+보정 구간)**

압력 저하 38.8%, 압력 과다 44.1%, 누설 66.7%를 잡고, 정밀도는 항상 경보의 1.7~2.7배다(오경보 주 2.0~2.6회). 놓침이 더 비싸다는 현장 조건을 반영한 **기본 운영안**이다. 하한의 기본 비율을 보정 구간만으로 잡으면 압력 저하가 35.8%로 조금 낮아지고, 나머지는 같다.

**한계**
- 사건이 적다(67·68·15건). 탐지율 35% 근처에서 67건이면 95% 신뢰구간이 약 ±11%p다. `field_eval`보다 1~2%p 높은 압력 저하·과다 차이는 우연과 구분되지 않는다. 누설 15건은 특히 불안정하다.
- 무작위 기대 탐지율은 경보 수만 맞춘 근사값이다(24시간 재경보 금지, 기록 공백은 반영하지 않음). 경보가 많을수록 실제보다 낮게 나온다.
- 장비마다 시간 분할 지점이 하나다. 분할 지점을 옮겨 가며(walk-forward) 평가하면 더 안정적이다.
- 하이퍼파라미터, 입력 조합은 EDA에서 정한 대로 한 번만 시험했다. 이 평가 구간을 보고 고르면 점수가 부풀려진다.
- 압력 저하 탐지기는 고장 이력 의존도가 커서, 고장 기록이 적은 새 장비에서는 약할 수 있다.

**다음 단계**
1. 경보 기준을 매주 최근 데이터로 다시 정하는 방식(재학습 파이프라인)을 시험한다.
2. 시간 분할 지점을 여러 개로 옮겨 가며 평가해 결과의 흔들림을 본다.
3. 코드를 `src/`의 함수로 옮겨 `retrain.py`로 돌릴 수 있게 한다.